# SpeedEstimator — MiDaS × YOLOv8 × ByteTrack × Kalman Filter

**Run on GPU runtime** (Runtime → Change runtime type → T4 GPU)

### Cells
1. **Cell 1** — Install dependencies
2. **Cell 2** — Camera calibration (focal length)
3. **Cell 3** — Main speed estimator (change `EXPERIMENT_MODE` per run)

### Experiment modes
| Mode | Description |
|---|---|
| `full` | Proposed system: geo-Z anchor + MiDaS modulation + Kalman filter |
| `geo_only` | Baseline A: bounding-box geometry only, no MiDaS |
| `no_kalman` | Baseline B: MiDaS fusion, no Kalman filter |

In [ ]:
# Cell 1 — Install dependencies
!pip install ultralytics supervision opencv-python torch torchvision
!apt-get install -y ffmpeg  # needed for H264 output

In [ ]:
# Cell 2 — Camera calibration
# Upload a short clip of a parked car at a known distance to compute focal length.
from google.colab import files
import cv2

print("Upload a short clip of a parked car at a known distance...")
uploaded = files.upload()
cap = cv2.VideoCapture(list(uploaded.keys())[0])
ret, frame = cap.read()
cap.release()
if not ret:
    raise RuntimeError("Could not read frame.")
cv2.imwrite("calibration_frame.jpg", frame)
files.download("calibration_frame.jpg")
print("Open the downloaded frame, measure the car's pixel width, fill in below, re-run this cell.")

# ---- Fill in after measuring ----
known_real_width_m   = 1.8    # metres (average car width)
known_distance_m     = 10.0   # metres (tape measure from camera to car)
observed_pixel_width = 320    # <- ONLY number you change

FOCAL_LENGTH_PX = (observed_pixel_width * known_distance_m) / known_real_width_m
print(f"Focal length set: {FOCAL_LENGTH_PX:.1f} px — ready for Cell 3")

In [ ]:
# Cell 3 — Main speed estimator
import cv2
import torch
import numpy as np
import supervision as sv
from ultralytics import YOLO
import collections
import subprocess, os, json
from google.colab import files
from google.colab.patches import cv2_imshow

# =========================================
# SAFETY CHECK
# =========================================
if 'FOCAL_LENGTH_PX' not in dir():
    raise RuntimeError("Run Cell 2 (calibration) first!")

# =========================================
# EXPERIMENT MODE  <- change this each run
# "full"      = geo_Z anchor + MiDaS modulation + Kalman (proposed system)
# "geo_only"  = bounding-box geometry only, no MiDaS          (Baseline A)
# "no_kalman" = geo_Z + MiDaS fusion, but no Kalman gate      (Baseline B)
# =========================================
EXPERIMENT_MODE = "full"

# =========================================
# CONFIG
# =========================================
VEHICLE_CLASS_IDS = [2, 3, 5, 6, 7]
REAL_WORLD_WIDTHS = {
    2: 1.8,  3: 0.8,  5: 2.5,  6: 3.2,  7: 2.4,
}
DEFAULT_WIDTH_M  = 1.8
SMOOTHING_WINDOW = 25
MIN_BOX_WIDTH_PX = 45
MIN_PIXEL_SPEED  = 3.0
MIN_SPEED_KMH    = 10.0
DEPTH_ALPHA      = 0.92
SPEED_ALPHA      = 0.90
MIDAS_MOD        = 0.06
MIDAS_SKIP       = 3        # run MiDaS every N frames
Z_MAX            = 40.0
Z_MIN            = 1.0
SPEED_CAP        = 130.0

# =========================================
# DEPTH FUNCTIONS
# =========================================
def get_geo_z(bbox_w, class_id, focal_px, cy, frame_h):
    real_w = REAL_WORLD_WIDTHS.get(int(class_id), DEFAULT_WIDTH_M)
    geo_z  = (focal_px * real_w) / max(bbox_w, 1)
    v_ratio   = cy / max(frame_h, 1)
    z_ceiling = Z_MAX * (1.0 - 0.6 * v_ratio)
    return float(np.clip(geo_z, Z_MIN, z_ceiling))

def fuse_depth(geo_z, midas_val, d_min, d_max):
    if d_max - d_min < 1e-6:
        return geo_z
    normalized = (midas_val - d_min) / (d_max - d_min)
    factor     = 1.0 + MIDAS_MOD * (0.5 - normalized)
    return float(np.clip(geo_z * factor, Z_MIN, Z_MAX))

# =========================================
# KALMAN FACTORY
# =========================================
def make_kalman(cx, cy):
    kf = cv2.KalmanFilter(4, 2)
    kf.transitionMatrix    = np.array([[1,0,1,0],[0,1,0,1],[0,0,1,0],[0,0,0,1]], np.float32)
    kf.measurementMatrix   = np.array([[1,0,0,0],[0,1,0,0]], np.float32)
    kf.processNoiseCov     = np.eye(4, dtype=np.float32) * 0.02
    kf.measurementNoiseCov = np.eye(2, dtype=np.float32) * 1.5
    kf.statePre = np.array([[cx],[cy],[0.],[0.]], np.float32)
    return kf

# =========================================
# SPEED FORMULA
# =========================================
def compute_speed_kmh(kf_vx, kf_vy, avg_z, history_z, fps, focal_px):
    dt   = 1.0 / fps
    dx_m = (kf_vx * avg_z) / focal_px
    dy_m = (kf_vy * avg_z) / focal_px
    dz_m = 0.0
    if len(history_z) >= 3:
        dz_m = (list(history_z)[-1] - list(history_z)[-3]) * 0.4
    dist_m    = np.sqrt(dx_m**2 + dy_m**2 + dz_m**2)
    speed_mps = dist_m / dt
    return float(np.clip(speed_mps * 3.6, 0.0, SPEED_CAP))

def compute_speed_raw_pixel(cx_prev, cy_prev, cx_cur, cy_cur, avg_z, history_z, fps, focal_px):
    dt   = 1.0 / fps
    dx_m = ((cx_cur - cx_prev) * avg_z) / focal_px
    dy_m = ((cy_cur - cy_prev) * avg_z) / focal_px
    dz_m = 0.0
    if len(history_z) >= 3:
        dz_m = (list(history_z)[-1] - list(history_z)[-3]) * 0.4
    dist_m    = np.sqrt(dx_m**2 + dy_m**2 + dz_m**2)
    speed_mps = dist_m / dt
    return float(np.clip(speed_mps * 3.6, 0.0, SPEED_CAP))

# =========================================
# OUTPUT HELPER
# =========================================
def make_writer(path, fps, w, h):
    return cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*"XVID"), fps, (w, h))

def encode_h264(raw, out, crf=20):
    subprocess.run(["ffmpeg","-y","-i",raw,"-c:v","libx264","-crf",str(crf),
                    "-preset","fast","-pix_fmt","yuv420p",out],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    os.remove(raw)

# =========================================
# SETUP MODELS
# =========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model = YOLO("yolov8n.pt")
print("YOLOv8n loaded")

use_midas = EXPERIMENT_MODE in ("full", "no_kalman")
midas = None
if use_midas:
    midas_type     = "MiDaS_small"
    midas          = torch.hub.load("intel-isl/MiDaS", midas_type).to(device).eval()
    transforms_hub = torch.hub.load("intel-isl/MiDaS", "transforms")
    transform      = (transforms_hub.small_transform
                      if midas_type == "MiDaS_small" else transforms_hub.dpt_transform)
    print(f"MiDaS loaded ({midas_type})")
else:
    print("MiDaS skipped (geo_only mode)")

# =========================================
# UPLOAD VIDEO
# =========================================
print(f"\nMode: {EXPERIMENT_MODE}")
print("Upload your traffic video...")
uploaded   = files.upload()
video_path = list(uploaded.keys())[0]
cap        = cv2.VideoCapture(video_path)
fps    = cap.get(cv2.CAP_PROP_FPS) or 30.0
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"Video: {width}x{height} @ {fps:.1f} FPS | {total} frames")
print(f"Focal length: {FOCAL_LENGTH_PX:.1f} px")

RAW_PATH = "raw_out.avi"
OUT_PATH = f"speed_{EXPERIMENT_MODE}.mp4"
out      = make_writer(RAW_PATH, fps, width, height)
tracker  = sv.ByteTrack(frame_rate=int(fps))

# Per-track state
kalman_filters  = {}
track_history_z = {}
prev_depth_ema  = {}
prev_speed_ema  = {}
prev_cx_cy      = {}

# Experiment logging
track_speed_log  = {}
track_class_log  = {}
phantom_readings = 0
total_readings   = 0

box_ann   = sv.BoxAnnotator(thickness=2)
label_ann = sv.LabelAnnotator(text_scale=0.5, text_thickness=1, text_padding=4)

# =========================================
# MAIN LOOP
# =========================================
frame_idx     = 0
preview_frame = None
depth_map     = None
d_min, d_max  = 0.0, 1.0
print("Processing...")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    frame_idx += 1
    if frame_idx % 60 == 0:
        print(f"  {frame_idx}/{total} ({100*frame_idx//max(total,1)}%)")

    # --- YOLO ---
    results    = model(frame, verbose=False)[0]
    detections = sv.Detections.from_ultralytics(results)
    detections = detections[np.isin(detections.class_id, VEHICLE_CLASS_IDS)]
    detections = tracker.update_with_detections(detections)

    # --- MiDaS every MIDAS_SKIP frames, reuse otherwise ---
    if use_midas and midas is not None and frame_idx % MIDAS_SKIP == 0:
        img_rgb     = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        input_batch = transform(img_rgb).to(device)
        with torch.no_grad():
            pred = midas(input_batch)
            pred = torch.nn.functional.interpolate(
                pred.unsqueeze(1), size=(height, width),
                mode="bicubic", align_corners=False).squeeze()
        depth_map = pred.cpu().numpy()
        d_min, d_max = depth_map.min(), depth_map.max()

    labels = []
    if len(detections) > 0 and detections.tracker_id is not None:
        for xyxy, class_id, tracker_id in zip(
            detections.xyxy, detections.class_id, detections.tracker_id
        ):
            x1, y1, x2, y2 = map(float, xyxy)
            bbox_w = x2 - x1
            tid    = int(tracker_id)

            if bbox_w < MIN_BOX_WIDTH_PX:
                labels.append("")
                continue

            cx = int(np.clip((x1+x2)/2, 0, width-1))
            cy = int(np.clip(y2,         0, height-1))

            # --- Geometric depth ---
            geo_z = get_geo_z(bbox_w, class_id, FOCAL_LENGTH_PX, cy, height)

            # --- Depth fusion ---
            if EXPERIMENT_MODE == "geo_only" or depth_map is None:
                refined_z = geo_z
            else:
                patch     = depth_map[max(0,cy-5):min(height,cy+5),
                                      max(0,cx-5):min(width, cx+5)]
                midas_val = float(np.median(patch)) if patch.size > 0 else 1.0
                refined_z = fuse_depth(geo_z, midas_val, d_min, d_max)

            # --- Depth EMA ---
            if tid in prev_depth_ema:
                refined_z = DEPTH_ALPHA * prev_depth_ema[tid] + (1-DEPTH_ALPHA) * refined_z
            prev_depth_ema[tid] = refined_z

            if tid not in track_history_z:
                track_history_z[tid] = collections.deque(maxlen=SMOOTHING_WINDOW)
            track_history_z[tid].append(refined_z)
            avg_z = float(np.mean(track_history_z[tid]))

            # --- Speed ---
            if EXPERIMENT_MODE == "no_kalman":
                if tid in prev_cx_cy:
                    px, py = prev_cx_cy[tid]
                    pixel_disp = np.sqrt((cx-px)**2 + (cy-py)**2)
                    if pixel_disp < MIN_PIXEL_SPEED:
                        raw_speed = 0.0
                    else:
                        raw_speed = compute_speed_raw_pixel(
                            px, py, cx, cy, avg_z, track_history_z[tid], fps, FOCAL_LENGTH_PX)
                else:
                    raw_speed = 0.0
                prev_cx_cy[tid] = (cx, cy)
            else:
                if tid not in kalman_filters:
                    kalman_filters[tid] = make_kalman(cx, cy)
                kf = kalman_filters[tid]
                kf.correct(np.array([[np.float32(cx)],[np.float32(cy)]]))
                kf_pred = kf.predict()
                kf_vx, kf_vy = float(kf_pred[2]), float(kf_pred[3])
                pixel_speed   = np.sqrt(kf_vx**2 + kf_vy**2)
                if pixel_speed < MIN_PIXEL_SPEED:
                    raw_speed = 0.0
                else:
                    raw_speed = compute_speed_kmh(
                        kf_vx, kf_vy, avg_z, track_history_z[tid], fps, FOCAL_LENGTH_PX)

            # --- Smoothing & floor ---
            prev_s   = prev_speed_ema.get(tid, raw_speed)
            smooth_s = SPEED_ALPHA * prev_s + (1-SPEED_ALPHA) * raw_speed
            prev_speed_ema[tid] = smooth_s
            display_speed = 0.0 if smooth_s < MIN_SPEED_KMH else smooth_s

            # --- Logging ---
            total_readings += 1
            if display_speed > 180.0:
                phantom_readings += 1
            if tid not in track_speed_log:
                track_speed_log[tid] = []
                track_class_log[tid] = model.names[int(class_id)]
            track_speed_log[tid].append(display_speed)

            class_name = model.names[int(class_id)]
            labels.append(f"[{class_name}] #{tid}  {display_speed:.0f} km/h")

    # --- Annotate ---
    annotated = box_ann.annotate(scene=frame.copy(), detections=detections)
    annotated = label_ann.annotate(scene=annotated, detections=detections, labels=labels)
    cv2.rectangle(annotated, (0,0), (520, 36), (0,0,0), -1)
    cv2.putText(annotated,
        f"Frame {frame_idx}/{total}  |  Mode: {EXPERIMENT_MODE}  |  {fps:.0f}fps",
        (8, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0,255,180), 1, cv2.LINE_AA)
    out.write(annotated)
    if frame_idx == 60:
        preview_frame = annotated.copy()

cap.release()
out.release()

encode_h264(RAW_PATH, OUT_PATH)

if preview_frame is not None:
    print("Preview (frame 60):")
    cv2_imshow(preview_frame)

# =========================================
# EXPERIMENT RESULTS
# =========================================
all_speeds = [s for spd in track_speed_log.values() for s in spd if s > 0]
if all_speeds:
    summary = {
        "mode": EXPERIMENT_MODE,
        "num_tracks": len(track_speed_log),
        "mean_speed_kmh": round(float(np.mean(all_speeds)), 2),
        "std_speed_kmh":  round(float(np.std(all_speeds)),  2),
        "phantom_rate_pct": round(100.0 * phantom_readings / max(total_readings, 1), 2),
        "total_measurements": len(all_speeds),
    }
    print("\n" + "="*60)
    print("EXPERIMENT RESULTS — copy this block and paste to chat")
    print("="*60)
    print(json.dumps(summary, indent=2))
    print("="*60)
else:
    print("No speed measurements recorded. Check MIN_BOX_WIDTH_PX or confidence threshold.")

print(f"\nDownloading {OUT_PATH}...")
files.download(OUT_PATH)